# Novel Grammar Induction

**Last updated:** 2026-04-09 — PT

**Track:** Learning | **Sub-Ability:** Language learning

Can the model induce grammar rules of a novel language from examples?
Pre/post learning framework: observe valid sentences, then judge grammaticality.

**Difficulty grid:** grammar complexity × num examples × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import sqlite3
import json
import time
import urllib.request
import urllib.error

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_yes_no(response: str) -> str:
    """Extract YES or NO from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    # Check lines from the end for a short YES/NO answer
    for line in reversed(lines):
        clean = line.strip('"\'').rstrip('.!?, ').lower().strip()
        if clean in ('yes', 'no'):
            return clean
    # Fallback: search entire text for last yes/no
    matches = re.findall(r'\b(yes|no)\b', text.lower())
    return matches[-1] if matches else ''

In [ ]:
# === Novel grammar engine ===
# Word classes with nonsense words
NOUNS = ['wug', 'blick', 'dax']
VERBS = ['fep', 'tup', 'niz']
ADJS  = ['ral', 'gom']
DETS  = ['sib', 'vex']

def starts_with_vowel(w):
    return w[0] in 'aeiou'

# --- Simple grammar: DET NOUN VERB (fixed word order, no agreement) ---
def gen_simple_sentences(rng, n):
    """Generate n valid sentences: DET NOUN VERB."""
    sents = set()
    while len(sents) < n:
        sents.add((rng.choice(DETS), rng.choice(NOUNS), rng.choice(VERBS)))
    return [' '.join(s) for s in sents]

def gen_simple_violations(rng, valid_sents):
    """Generate violations: wrong word order."""
    tests = []
    # Valid test sentence
    s = (rng.choice(DETS), rng.choice(NOUNS), rng.choice(VERBS))
    tests.append((' '.join(s), 'yes'))
    # NOUN DET VERB (swap det/noun)
    s2 = (rng.choice(NOUNS), rng.choice(DETS), rng.choice(VERBS))
    tests.append((' '.join(s2), 'no'))
    # VERB NOUN DET (reversed)
    s3 = (rng.choice(VERBS), rng.choice(NOUNS), rng.choice(DETS))
    tests.append((' '.join(s3), 'no'))
    # DET VERB NOUN (verb before noun)
    s4 = (rng.choice(DETS), rng.choice(VERBS), rng.choice(NOUNS))
    tests.append((' '.join(s4), 'no'))
    return tests

# --- Medium grammar: DET NOUN VERB with agreement ---
# Rule: "sib" pairs with consonant-initial nouns, "vex" with vowel-initial
# Verb agreement: if det is "sib", verb takes suffix "-a"; if "vex", verb takes suffix "-o"
def medium_det_for_noun(noun):
    return 'vex' if starts_with_vowel(noun) else 'sib'

def medium_verb_form(verb, det):
    return verb + 'a' if det == 'sib' else verb + 'o'

def gen_medium_sentences(rng, n):
    """Generate n valid sentences: DET NOUN VERB+suffix with agreement."""
    sents = set()
    while len(sents) < n:
        noun = rng.choice(NOUNS)
        det = medium_det_for_noun(noun)
        verb = rng.choice(VERBS)
        vf = medium_verb_form(verb, det)
        sents.add((det, noun, vf))
    return [' '.join(s) for s in sents]

def gen_medium_violations(rng, valid_sents):
    """Generate violations: agreement errors."""
    tests = []
    # Valid
    noun = rng.choice(NOUNS)
    det = medium_det_for_noun(noun)
    verb = rng.choice(VERBS)
    vf = medium_verb_form(verb, det)
    tests.append((f'{det} {noun} {vf}', 'yes'))
    # Wrong determiner (use opposite)
    noun2 = rng.choice(NOUNS)
    wrong_det = 'vex' if medium_det_for_noun(noun2) == 'sib' else 'sib'
    verb2 = rng.choice(VERBS)
    vf2 = medium_verb_form(verb2, wrong_det)  # verb agrees with wrong det
    tests.append((f'{wrong_det} {noun2} {vf2}', 'no'))
    # Wrong verb suffix (det is correct but verb suffix mismatches)
    noun3 = rng.choice(NOUNS)
    det3 = medium_det_for_noun(noun3)
    verb3 = rng.choice(VERBS)
    wrong_suffix = 'o' if det3 == 'sib' else 'a'
    tests.append((f'{det3} {noun3} {verb3}{wrong_suffix}', 'no'))
    # Wrong word order
    noun4 = rng.choice(NOUNS)
    det4 = medium_det_for_noun(noun4)
    verb4 = rng.choice(VERBS)
    vf4 = medium_verb_form(verb4, det4)
    tests.append((f'{noun4} {det4} {vf4}', 'no'))
    return tests

# --- Complex grammar: DET ADJ NOUN VERB (DET NOUN) — nested object clause ---
# Same det-noun agreement as medium, plus:
# - ADJ must precede NOUN
# - Object clause: (DET NOUN) with its own agreement
# - Verb suffix: "-a"/"-o" based on SUBJECT det (not object det)
def gen_complex_sentence(rng):
    """Generate one valid complex sentence: DET ADJ NOUN VERB(suffix) DET NOUN."""
    subj_noun = rng.choice(NOUNS)
    subj_det = medium_det_for_noun(subj_noun)
    adj = rng.choice(ADJS)
    verb = rng.choice(VERBS)
    vf = medium_verb_form(verb, subj_det)
    obj_noun = rng.choice(NOUNS)
    obj_det = medium_det_for_noun(obj_noun)
    return (subj_det, adj, subj_noun, vf, obj_det, obj_noun)

def complex_to_str(parts):
    return ' '.join(parts)

def gen_complex_sentences(rng, n):
    """Generate n valid complex sentences."""
    sents = set()
    while len(sents) < n:
        sents.add(gen_complex_sentence(rng))
    return [complex_to_str(s) for s in sents]

def gen_complex_violations(rng, valid_sents):
    """Generate violations: agreement, order, nesting errors."""
    tests = []
    # Valid
    s = gen_complex_sentence(rng)
    tests.append((complex_to_str(s), 'yes'))
    # Wrong subject det-noun agreement
    s2 = list(gen_complex_sentence(rng))
    s2[0] = 'vex' if s2[0] == 'sib' else 'sib'  # flip subject det
    tests.append((complex_to_str(s2), 'no'))
    # Wrong object det-noun agreement
    s3 = list(gen_complex_sentence(rng))
    s3[4] = 'vex' if s3[4] == 'sib' else 'sib'  # flip object det
    tests.append((complex_to_str(s3), 'no'))
    # Missing adjective (DET NOUN VERB DET NOUN — wrong structure)
    s4 = gen_complex_sentence(rng)
    tests.append((f'{s4[0]} {s4[2]} {s4[3]} {s4[4]} {s4[5]}', 'no'))
    return tests

# === Build dataset: complexity × evidence × seeds ===
COMPLEXITY = ['simple', 'medium', 'complex']
EVIDENCE = [4, 8, 12]   # few, mid, many example sentences
SEEDS = 3
rows = []
tid = 0

gen_funcs = {
    'simple':  (gen_simple_sentences, gen_simple_violations),
    'medium':  (gen_medium_sentences, gen_medium_violations),
    'complex': (gen_complex_sentences, gen_complex_violations),
}

for complexity in COMPLEXITY:
    gen_sents, gen_viols = gen_funcs[complexity]
    for n_ex in EVIDENCE:
        for seed in range(SEEDS):
            rng = random.Random(seed * 100 + EVIDENCE.index(n_ex) * 10 + COMPLEXITY.index(complexity))
            valid_examples = gen_sents(rng, n_ex)
            test_items = gen_viols(rng, valid_examples)
            material = '\n'.join(f'  {s}' for s in valid_examples)
            label = f'{complexity}_{n_ex}ex'
            for test_sent, expected_ans in test_items:
                rows.append({
                    'task_id': tid,
                    'seed': seed,
                    'complexity': complexity,
                    'evidence': f'{n_ex}_examples',
                    'difficulty_label': label,
                    'material': material,
                    'test_sentence': test_sent,
                    'expected': expected_ans,
                    'item_desc': f'{complexity} grammar, {n_ex} examples, seed {seed}',
                })
                tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items ({len(dataset)//4} conditions x 4 test items each)')
print(f'Complexity levels: {COMPLEXITY}')
print(f'Evidence levels: {EVIDENCE}')
print()
for _, row in dataset.iterrows():
    print(f'  [{row["difficulty_label"]:12s}] test="{row["test_sentence"]}"  expected={row["expected"]}')


In [ ]:
DB_PATH = 'grammar_induction_runs.db'
db = sqlite3.connect(DB_PATH)
db.execute('DROP TABLE IF EXISTS runs')
db.execute('''
    CREATE TABLE runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, model TEXT, task_id TEXT,
        complexity TEXT, evidence TEXT, difficulty_label TEXT,
        seed INTEGER, item_desc TEXT,
        test_sentence TEXT, expected TEXT,
        pre_raw TEXT, pre_extracted TEXT, pre_correct INTEGER,
        study_notes TEXT,
        post_raw TEXT, post_extracted TEXT, post_correct INTEGER,
        pre_time_s REAL, study_time_s REAL, post_time_s REAL
    )
''')
db.commit()
print(f'SQLite ready (fresh): {DB_PATH}')

In [ ]:
# === Prompt templates ===

PRE_P = """Here are valid sentences in a novel language:
{material}

Is this sentence grammatical in the same language?
  "{test_sentence}"

Reply with ONLY YES or NO."""

STUDY_P = """Here are valid sentences in a novel language:
{material}

Your task: Analyze the grammar of this language thoroughly.
1. Identify the word classes (which words are nouns, verbs, determiners, adjectives, etc.).
2. Determine the word order rules.
3. Identify any agreement patterns (e.g., which determiners go with which nouns, verb suffixes).
4. Check for embedding or nesting patterns.
5. Verify your grammar against EVERY example sentence.

Write a complete, precise grammar description."""

POST_P = """You previously studied a novel language and wrote these grammar notes:

--- YOUR GRAMMAR NOTES ---
{notes}
--- END NOTES ---

Here are the original valid sentences for reference:
{material}

Is this sentence grammatical in the same language?
  "{test_sentence}"

Reply with ONLY YES or NO."""

## Evaluation

In [ ]:
@kbench.task(name='novel_grammar_induction',
             description='Pre/post learning grammar induction from novel language examples')
def novel_grammar_induction(llm, material: str, test_sentence: str, expected: str,
                            complexity: str, evidence: str, difficulty_label: str,
                            seed: int, item_desc: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{complexity}_{evidence}_{seed}'

    # --- Pre-learning: judge grammaticality without study ---
    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material, test_sentence=test_sentence))
    pre_time = time.time() - t0
    pre_extracted = extract_yes_no(pre_raw)
    pre_correct = pre_extracted == expected

    # --- Study phase: analyze the grammar ---
    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    # --- Post-learning: judge with notes ---
    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_sentence=test_sentence))
    post_time = time.time() - t0
    post_extracted = extract_yes_no(post_raw)
    post_correct = post_extracted == expected

    try:
        db.execute(
            """INSERT INTO runs (timestamp,model,task_id,complexity,evidence,difficulty_label,
               seed,item_desc,test_sentence,expected,pre_raw,pre_extracted,pre_correct,
               study_notes,post_raw,post_extracted,post_correct,pre_time_s,study_time_s,post_time_s)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (ts,model_name,tid,complexity,evidence,difficulty_label,int(seed),item_desc,
             test_sentence,expected,pre_raw,pre_extracted,int(pre_correct),
             notes,post_raw,post_extracted,int(post_correct),pre_time,study_time,post_time))
        db.commit()
    except Exception as e: print(f'  [SQLite warn] {e}')

    b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
    print(f'[{difficulty_label:12s}] "{test_sentence}"  exp={expected}  pre="{pre_extracted}"({b})  post="{post_extracted}"({l})  times: {pre_time:.1f}s/{study_time:.1f}s/{post_time:.1f}s')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

runs = novel_grammar_induction.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                        n_jobs=1, timeout=3600, max_attempts=1)
print(f'\nDone: {len(runs.as_dataframe())} items')

## Results

In [ ]:
results = pd.read_sql('SELECT * FROM runs ORDER BY task_id', db)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc
room = 1.0 - pre_acc
eff = gain / room if room > 0 else 0.0

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print(f'Learning efficiency:    {eff:.1%}')
print()

# Per difficulty label
print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per item
print()
print('Per-item detail:')
print('-' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    delta = '  HELPED' if not row['pre_correct'] and row['post_correct'] else \
            '  HURT' if row['pre_correct'] and not row['post_correct'] else ''
    print(f'  [{row["difficulty_label"]:12s}] "{row["test_sentence"]}"  exp={row["expected"]}  '
          f'pre="{row["pre_extracted"]}"({b})  post="{row["post_extracted"]}"({l}){delta}')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Test: "{row["test_sentence"]}"  Expected: {row["expected"]}')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Pre vs Post-Learning: Grammar Induction')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('grammar_induction_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Post-run: upload to BigQuery + Google Sheets via REST API ===
# Safe to do after benchmark — SDK can break, doesn't matter.

results_upload = pd.read_sql('SELECT * FROM runs', db)
db.close()
print(f'SQLite closed. {len(results_upload)} rows to upload.\n')

auth_token = None
gcp_project = None
try:
    import google.auth
    import google.auth.transport.requests
    creds, project = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    auth_token = creds.token
    gcp_project = project
    print(f'Authenticated. Project: {gcp_project}')
except Exception as e:
    print(f'Google auth not available: {e}')

BQ_DATASET = 'benchmark_runs'

# --- BigQuery ---
if auth_token and gcp_project:
    BQ_TABLE = DB_PATH.replace('_runs.db', '')
    try:
        import urllib.parse
        ds_url = f'https://api.kaggle.com/api/v1/bigquery/projects/{gcp_project}/datasets'
        ds_body = json.dumps({'datasetReference': {'datasetId': BQ_DATASET}, 'location': 'US'}).encode()
        req = urllib.request.Request(ds_url, data=ds_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        schema_fields = [{'name': c, 'type': 'STRING' if results_upload[c].dtype == 'object' else
                          'INTEGER' if 'correct' in c or c in ('seed','id') else 'FLOAT'}
                         for c in results_upload.columns]
        create_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables'
        create_body = json.dumps({'tableReference': {'projectId': gcp_project, 'datasetId': BQ_DATASET, 'tableId': BQ_TABLE},
                                  'schema': {'fields': schema_fields}}).encode()
        req = urllib.request.Request(create_url, data=create_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        table_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables/{BQ_TABLE}/insertAll'
        rows_json = [{'json': {c: str(v) if pd.notna(v) else '' for c, v in row.items()}} for _, row in results_upload.iterrows()]
        insert_body = json.dumps({'rows': rows_json}).encode()
        req = urllib.request.Request(table_url, data=insert_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'BigQuery: {len(results_upload)} rows -> {gcp_project}.{BQ_DATASET}.{BQ_TABLE}')
    except Exception as e:
        print(f'BigQuery failed (non-fatal): {e}')

# --- Google Sheets ---
if auth_token:
    SHEET_NAME = f'Learning Bench \u2014 {BQ_TABLE} Runs'
    try:
        import urllib.parse
        search_url = ('https://www.googleapis.com/drive/v3/files'
                      f'?q=name%3D%27{urllib.parse.quote(SHEET_NAME)}%27+and+mimeType%3D%27application/vnd.google-apps.spreadsheet%27'
                      '&fields=files(id,webViewLink)')
        req = urllib.request.Request(search_url, headers={'Authorization': f'Bearer {auth_token}'})
        resp = json.loads(urllib.request.urlopen(req).read())
        files = resp.get('files', [])
        if files:
            sid = files[0]['id']
        else:
            create_url = 'https://sheets.googleapis.com/v4/spreadsheets'
            req = urllib.request.Request(create_url, data=json.dumps({'properties': {'title': SHEET_NAME}}).encode(),
                                         method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            resp = json.loads(urllib.request.urlopen(req).read())
            sid = resp['spreadsheetId']
            header = list(results_upload.columns)
            body = json.dumps({'values': [header]}).encode()
            req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                         data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            urllib.request.urlopen(req)

        data_rows = [[str(v) if pd.notna(v) else '' for v in row] for _, row in results_upload.iterrows()]
        body = json.dumps({'values': data_rows}).encode()
        req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                     data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'Sheets: {len(results_upload)} rows -> "{SHEET_NAME}"')
    except Exception as e:
        print(f'Sheets failed (non-fatal): {e}')

print(f'\nDone. SQLite: {DB_PATH}')